# Vibration spectrum over time -- rekon10

One flight, every accelerometer on the vehicle, as a spectrogram against the FC clock, with
the ESC-RPM lines drawn on top so a peak can be read as "motor N, rev / blade-pass / 3rd"
rather than as an unexplained frequency. The throttle, RPM and current panels below share
the time axis: the lines should *track* them, and where they do not is the interesting part.

Instruments, and what each can reach:

| instrument | rate | Nyquist | mount |
|---|---|---|---|
| FC `ACC` (x2) | ~990 Hz | ~495 Hz | FC board, on isolation bobbins |
| campod `accel-camera` | fitted, ~3260 Hz | ~1630 Hz | beside the camera module |
| campod `accel-arm` | fitted, ~3175 Hz | ~1590 Hz | arm end of the same pod |

**Every campod rate is fitted per sensor per capture, never taken from the header.** The two
parts' clocks differ by ~2.8% and the datasheet specifies no tolerance
(`docs/rekon10/vibration-testing.md` sec. 2). The header's `odr_hz_nominal` is what was asked
for, not what happened.

Where things live: `docs/flight-data-layout.md`. What the streams mean and which clock each
is on: `docs/flight-data-interpretation.md`.

In [ ]:
input_file = "/home/jovyan/datasets/flights/rekon10/260923-new-props/fc/fc-log-id3.bin"
flight_dir = ""     # "" -> derived from input_file (handles the log sitting in <flight>/fc/)
out_dir = ""        # "" -> <flight_dir>/derived
fmax_hz = 600.0     # top of the plotted band; rev/blade-pass/3rd all live below this
debug = False

In [ ]:
import json, os, sys, math
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from scipy.signal import spectrogram, correlate

REPO = Path.cwd()
while not (REPO / "analysis" / "ardupilot_log.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "analysis"))
from ardupilot_log import parse_log

LOG = Path(input_file)
FLIGHT = Path(flight_dir) if flight_dir else (LOG.parent.parent if LOG.parent.name == "fc" else LOG.parent)
OUT = Path(out_dir) if out_dir else (FLIGHT / "derived")
OUT.mkdir(parents=True, exist_ok=True)
print("log        :", LOG)
print("flight dir :", FLIGHT)
print("out dir    :", OUT)

# Categorical slots, fixed order, never cycled (dataviz: colour follows the entity, not its rank).
# Contrast against a light surface is under 3:1 for slots 3-4, so every motor trace is also
# direct-labelled and every series is repeated in the emitted JSON table.
MOTOR_COLORS = {"Motor1": "#2a78d6", "Motor2": "#eb6834", "Motor3": "#1baf7a", "Motor4": "#eda100"}
# cividis: monotonic in lightness and CVD-safe. Not a rainbow -- a rainbow ramp invents
# structure in a spectrogram by putting hue edges at arbitrary power levels.
SPEC_CMAP = "cividis"
GRID = dict(color="#d9d8d2", lw=0.6, alpha=0.9)

## 1. The FC log: window, motor mapping, predicted lines

`ESC.Instance` is the **output channel, not the ArduPilot motor number** -- it has to be mapped
through `SERVOn_FUNCTION` or a fore/aft split reads as a diagonal one
([#169](https://github.com/symmatree/coordinator/issues/169)). On rekon10 the map is not the
identity, so this is not optional.

In [ ]:
F, fc_duration, parms = parse_log(LOG, ["ESC", "ACC", "GYR", "CTUN", "ARM", "MODE", "VIBE", "BAT", "RCOU", "XKF1", "GPS", "PARM"])
A = F["ARM"]
T_ARM = float(A[A.ArmState == 1].t_s.iloc[0])
T_DIS = float(A[A.ArmState == 0].t_s.iloc[0])

# Liftoff/touchdown, derived, ARM-GATED. capture_align.airborne_window() is not used here: its
# liftoff search is not gated on t > t_arm, so a pre-arm altitude excursion (the vehicle being
# carried, or baro drift across a long ground hold) makes it return a liftoff BEFORE arming --
# it does on this flight, by 50.9 s. The explicit ARM record is the cross-check that catches it.
xk = F["XKF1"]; xk = xk[xk.C == 0].sort_values("t_s")
ct = F["CTUN"].sort_values("t_s")
alt = np.interp(ct.t_s.values, xk.t_s, -xk.PD.values)
ground = float(np.median(alt[(ct.t_s.values > T_ARM - 30) & (ct.t_s.values < T_ARM)]))
up = np.where((ct.t_s.values > T_ARM) & (alt > ground + 0.30))[0]
on = np.where((ct.t_s.values > T_ARM) & (ct.ThO.values > 0))[0]
LIFT = float(ct.t_s.values[up[0]]) if len(up) else T_ARM
LAND = float(ct.t_s.values[on[-1]]) if len(on) else T_DIS
print(f"ARM {T_ARM:.2f}  liftoff {LIFT:.2f} (+{LIFT-T_ARM:.2f} s)  touchdown {LAND:.2f}  DISARM {T_DIS:.2f}")

MOTOR_FN = {33: "Motor1", 34: "Motor2", 35: "Motor3", 36: "Motor4"}
chan2motor = {}
for k, v in parms.items():
    if k.startswith("SERVO") and k.endswith("_FUNCTION") and int(v) in MOTOR_FN:
        chan2motor[int(k.split("_")[0][5:])] = MOTOR_FN[int(v)]
inst2motor = {ch - 1: m for ch, m in chan2motor.items()}
# ArduPilot quad-X: 1=front-right 2=rear-left 3=front-left 4=rear-right
MOTOR_POS = {"Motor1": "front-right", "Motor2": "rear-left", "Motor3": "front-left", "Motor4": "rear-right"}
print("ESC Instance -> motor:", {k: f"{v} ({MOTOR_POS[v]})" for k, v in sorted(inst2motor.items())})
assert len(inst2motor) >= 1, "no SERVOn_FUNCTION motor assignments in PARM -- cannot label lines"

esc = F["ESC"]
esc_by_motor = {inst2motor[int(i)]: esc[esc.Instance == i].sort_values("t_s")
                for i in sorted(esc.Instance.unique()) if int(i) in inst2motor}
hov = {m: d[(d.t_s >= T_ARM + 15) & (d.t_s <= T_DIS - 15)] for m, d in esc_by_motor.items()}
print("\nhover-window means (arm+15 .. disarm-15):")
for m in sorted(hov):
    d = hov[m]
    print(f"  {m} ({MOTOR_POS[m]:11s}) rpm {d.RPM.mean():6.0f} +/- {d.RPM.std():4.0f}"
          f"  -> rev {d.RPM.mean()/60:6.2f} Hz  blade-pass {d.RPM.mean()/30:6.2f} Hz"
          f"  curr {d.Curr.mean():5.2f} A")
front = np.mean([hov[m].RPM.mean() for m in hov if MOTOR_POS[m].startswith("front")])
rear = np.mean([hov[m].RPM.mean() for m in hov if MOTOR_POS[m].startswith("rear")])
print(f"  front {front:.0f} vs rear {rear:.0f} -> front/rear split {(front/rear-1)*100:+.1f}%  (#169)")
# The FC's own absolute time, from its GPS week/ms. Needed only to express the pod's wall-clock
# join in FC time so the two joins can be COMPARED; the alignment itself is done by motion below,
# so an error here lands in the reported correction rather than in the result.
# GWk > 1000 rejects the pre-lock rows (flight-data-interpretation.md).
import datetime as _dt
GPS_EPOCH = _dt.datetime(1980, 1, 6, tzinfo=_dt.timezone.utc)
GPS_UTC_LEAP_S = 18       # GPS-UTC offset, constant since 2017-01-01
_g = F["GPS"]; _g = _g[(_g.I == 0) & (_g.GWk > 1000)].sort_values("t_s")
assert len(_g) > 10, "no GPS rows with GWk>1000 -- cannot anchor FC time to UTC"
_utc = np.array([(GPS_EPOCH + _dt.timedelta(weeks=int(w), milliseconds=int(ms))).timestamp() - GPS_UTC_LEAP_S
                 for w, ms in zip(_g.GWk.values, _g.GMS.values)])
_sl, _ic = np.polyfit(_g.t_s.values, _utc, 1)
_res = _utc - (_ic + _sl * _g.t_s.values)
UTC_ARM_UNIX = float(_ic + _sl * T_ARM)
print(f"\nFC t_s -> UTC fit over {len(_g)} GPS rows: residual rms {_res.std()*1000:.1f} ms, "
      f"slope {_sl:.9f}")
print(f"  ARM at {_dt.datetime.fromtimestamp(UTC_ARM_UNIX, _dt.timezone.utc).isoformat()}  "
      f"(unix {UTC_ARM_UNIX:.3f})")

### The source is not stationary, and that sets the resolution floor

A fixed-frequency spectrum over a long window smears each line across the RPM excursion in
that window. A bench fan is a constant-speed source; a closed-loop hover is not. This is why
a spectro**gram** is the right form here and a single long PSD is not -- and it bounds how
well any instrument can be expected to agree with the RPM prediction.

In [ ]:
print("rev-line wander over the armed window:")
for m in sorted(hov):
    r = hov[m].RPM / 60.0
    print(f"  {m}: rev {r.mean():6.2f} Hz  sd {r.std():4.2f} Hz  full spread {r.max()-r.min():5.2f} Hz")
print("\n=> expect agreement with the prediction at roughly the per-column rev-line sd, no better.")

## 2. campod accelerometers: load, fit the rate, place every sample honestly

Three things this does deliberately:

1. **Tolerates a truncated final record.** The writer `fsync`s at 1 Hz behind a 128 KiB
   writeback kick, so a power cut leaves the last partial record unparseable. That is one bad
   line, not a bad file -- it is counted and skipped, never silently ignored.
2. **Times samples from their own batch's timestamp**, not from a running sample counter.
   The cumulative index counts what was *read*, so it cannot see FIFO loss
   (`vibration-testing.md` sec. 2); the batch timestamps can. Each batch's `n` samples are placed
   backwards from its `boot_ns` at the fitted rate, which puts a discarded-sample hole where
   it belongs instead of smearing it through the series.
3. **Never bridges a hole.** Runs are split wherever consecutive samples are more than 3
   sample-periods apart, and each run is transformed separately.

In [ ]:
RAIL = 4095   # 13-bit signed, FULL_RES

def load_accel(path):
    hdr, bad = None, 0
    bt, bi, bn, ovr, xs, ys, zs, clk = [], [], [], [], [], [], [], []
    with open(path) as fh:
        for line in fh:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                bad += 1
                continue
            t = r.get("t") or r.get("type")
            if t == "header":
                hdr = r
            elif t == "clk":
                clk.append((r["boot_ns"], r["wall_ns"]))
            elif t == "b":
                bt.append(r["boot_ns"]); bi.append(r["i"]); bn.append(r["n"]); ovr.append(bool(r.get("ovr")))
                xs.append(r["x"]); ys.append(r["y"]); zs.append(r["z"])
    X = np.concatenate([np.asarray(v, float) for v in xs])
    Y = np.concatenate([np.asarray(v, float) for v in ys])
    Z = np.concatenate([np.asarray(v, float) for v in zs])
    bt = np.asarray(bt, float); bi = np.asarray(bi, float); bn = np.asarray(bn, int)
    rate = float(np.polyfit(bt / 1e9, bi, 1)[0])          # samples per second, fitted
    # per-sample boot time, placed backwards from each batch's own stamp
    tb = np.concatenate([bt[j] / 1e9 - (bn[j] - 1 - np.arange(bn[j])) / rate for j in range(len(bn))])
    clk = np.asarray(clk, dtype=np.int64)
    return dict(hdr=hdr, bad=bad, rate=rate, t_boot=tb, x=X, y=Y, z=Z,
                bn=bn, ovr=np.asarray(ovr), clk=clk,
                wall_minus_boot=float(np.median(clk[:, 1] - clk[:, 0])) / 1e9 if len(clk) else np.nan)

sessions = sorted((FLIGHT / "captures").glob("campod-*/*/"))
accels = {}
for sdir in sessions:
    for f in sorted(sdir.glob("accel-*.jsonl")):
        d = load_accel(f)
        key = f"{sdir.parent.name}/{d['hdr']['label']}"
        accels[key] = d
        g = d["hdr"]["scale_mg_per_lsb"] / 1000.0
        rail = ((np.abs(d["x"]) >= RAIL) | (np.abs(d["y"]) >= RAIL) | (np.abs(d["z"]) >= RAIL))
        print(f"{key:24s} nominal {d['hdr']['odr_hz_nominal']} Hz -> FITTED {d['rate']:7.2f} Hz"
              f"  ({(d['rate']/d['hdr']['odr_hz_nominal']-1)*100:+.2f}%)  samples {len(d['x']):,}"
              f"  truncated-tail lines skipped {d['bad']}"
              f"  self-test {'PASS' if d['hdr']['self_test']['all_pass'] else 'FAIL'}"
              f"  rail-hit {rail.mean()*100:.3f}%  separation_m {d['hdr']['separation_m']!r}")
assert accels, f"no campod accel jsonl under {FLIGHT/'captures'}"

ratios = sorted(accels)
if len(ratios) == 2:
    a, b = accels[ratios[0]]["rate"], accels[ratios[1]]["rate"]
    print(f"\npart-to-part clock ratio {max(a,b)/min(a,b):.6f} ({(max(a,b)/min(a,b)-1)*100:.4f}%)"
          f" -- a property of the two parts; compare it across captures to check the chain.")

### Putting the pod on the FC clock, by motion rather than by trusting either wall clock

The pod has no RTC and the coordinator's clock stepped **+85,093 s** three minutes before
arming on this flight, so a wall-clock join is not self-validating: the absence of a step
proves nothing and the presence of one this large proves the clock was wrong until it landed.

So the offset is *measured*: cross-correlate the pod's broadband vibration envelope against
the FC's own commanded throttle. Motors-on is a step both see. The wall-clock join is then
computed too, and **the two are compared** -- agreement is the check on the whole chain, the
same way the explicit-vs-derived liftoff check is.

In [ ]:
ENV_HZ = 10.0
LAG_SEARCH_S = 20.0
MIN_NCC = 0.35

def envelope(t, v, t0, t1, hz=ENV_HZ):
    """Peak-in-bin envelope. Empty bins are filled from their neighbours, not left NaN:
    np.maximum against a NaN seed propagates NaN through the whole array, which silently
    produces an all-zero envelope and a correlation of exactly 0."""
    edges = np.arange(t0, t1, 1.0 / hz)
    k = np.clip(np.digitize(t, edges) - 1, 0, len(edges) - 1)
    out = np.zeros(len(edges))
    seen = np.zeros(len(edges), bool)
    np.maximum.at(out, k, np.abs(v))
    seen[np.unique(k)] = True
    if not seen.all():
        idx = np.flatnonzero(seen)
        assert len(idx), "envelope is empty -- no samples in the window"
        out = np.interp(np.arange(len(edges)), idx, out[idx])
    return edges, out

# FC reference: commanded throttle, which is what the operator and the mixer both see
fc_t0, fc_t1 = T_ARM - 30, min(T_DIS + 10, fc_duration)
ct_w = ct[(ct.t_s >= fc_t0) & (ct.t_s <= fc_t1)]
ref_e, ref = envelope(ct_w.t_s.values, ct_w.ThO.values, fc_t0, fc_t1)
ref = np.nan_to_num(ref)
ref = (ref - ref.mean()) / (ref.std() or 1)

align = {}
for key, d in accels.items():
    mag = np.sqrt(d["x"] ** 2 + d["y"] ** 2 + d["z"] ** 2)
    # wall-clock join: boot -> unix -> FC t_s, using the FC's own GPS-derived UTC at ARM
    t_fc_wall = d["t_boot"] + d["wall_minus_boot"] - UTC_ARM_UNIX + T_ARM
    e, env = envelope(t_fc_wall, mag - np.median(mag), fc_t0, fc_t1)
    env = np.nan_to_num(env)
    env = (env - env.mean()) / (env.std() or 1)
    c = correlate(env, ref, mode="full") / len(ref)
    lags = (np.arange(len(c)) - (len(ref) - 1)) / ENV_HZ
    ok = np.abs(lags) <= LAG_SEARCH_S
    lag = float(lags[ok][np.argmax(c[ok])])
    peak = float(c[ok].max())
    align[key] = dict(lag_s=lag, ncc=peak, t_fc=t_fc_wall - lag)
    print(f"{key:24s} wall-clock join needs {lag:+.2f} s correction (NCC {peak:.3f}) to match FC throttle")
    # Fail loudly. A steady hover hides a clock error: shift the pod by 20 s and the line
    # FREQUENCIES barely move, so every downstream check still passes while the data is
    # misplaced. The alignment has to be shown to have worked, not assumed.
    assert peak >= MIN_NCC, (f"{key}: motion alignment failed (NCC {peak:.3f} < {MIN_NCC}) -- "
                            "the pod cannot be placed on the FC clock, so nothing below is safe")
    assert abs(abs(lag) - LAG_SEARCH_S) > 0.5, (f"{key}: best lag {lag:+.2f} s sits on the search "
                            f"bound +/-{LAG_SEARCH_S} s -- the true offset is outside the window")
spread = max(a["lag_s"] for a in align.values()) - min(a["lag_s"] for a in align.values())
print(f"\nspread across sensors on the same pod: {spread:.2f} s"
      f"  (they share a host clock, so this is the method's own resolution, not a disagreement)")
print(f"residual wall-clock error: the pod's own clock was off by {np.mean([a['lag_s'] for a in align.values()]):+.2f} s.")
print("The pod has no RTC and the coordinator's clock stepped +85,093 s three minutes before arming,")
print("so this is measured rather than assumed -- and a motors-on step is a physical event both")
print("instruments record, which no clock claim can contradict.")

## 3. Spectrograms

Each instrument on its own rate and its own resolution; all four on one shared FC time axis.
The ESC rev and blade-pass lines are drawn over the top from the RPM telemetry -- these are
*predictions*, not fits, so where a bright ridge sits off a line, that is the measurement
disagreeing with the prediction and worth reading.

Two honesty strips sit under each campod panel: the **clipped fraction** per column (the arm
sensor reaches the +/-16 g rail, and a clipped column's harmonics are partly the clipping) and
the **gaps** where a run was broken.

In [ ]:
def runs_of(t, rate, max_gap_periods=3.0):
    br = np.where(np.diff(t) > max_gap_periods / rate)[0]
    return list(zip(np.r_[0, br + 1], np.r_[br + 1, len(t)]))

def spec_of(t_fc, sig, rate, nperseg, fmax):
    cols_f, cols_t, cols_S, gaps = None, [], [], []
    for a, b in runs_of(t_fc, rate):
        if b - a < nperseg:
            if b - a > 1: gaps.append((float(t_fc[a]), float(t_fc[b - 1])))
            continue
        s = sig[a:b] - sig[a:b].mean()
        f, tt, S = spectrogram(s, fs=rate, nperseg=nperseg, noverlap=nperseg // 2,
                               window="hann", scaling="density", mode="psd")
        m = f <= fmax
        cols_f = f[m]
        cols_t.append(tt + t_fc[a]); cols_S.append(S[m])
    if not cols_S:
        return None
    order = np.argsort(np.concatenate(cols_t))
    return cols_f, np.concatenate(cols_t)[order], np.concatenate(cols_S, axis=1)[:, order], gaps

panels = []
for inst in sorted(F["ACC"].I.unique()):
    a = F["ACC"]; a = a[(a.I == inst) & (a.t_s >= fc_t0) & (a.t_s <= fc_t1)].sort_values("t_s")
    rate = len(a) / (a.t_s.max() - a.t_s.min())
    mag = np.sqrt(a.AccX.values ** 2 + a.AccY.values ** 2 + a.AccZ.values ** 2)
    sp = spec_of(a.t_s.values, mag, rate, 1024, fmax_hz)
    panels.append((f"FC ACC{int(inst)}  ({rate:.0f} Hz, Nyq {rate/2:.0f})", sp, None, None, rate))

for key, d in accels.items():
    t_fc = align[key]["t_fc"]
    g = d["hdr"]["scale_mg_per_lsb"] / 1000.0 * 9.80665
    m = (t_fc >= fc_t0) & (t_fc <= fc_t1)
    mag = np.sqrt(d["x"][m] ** 2 + d["y"][m] ** 2 + d["z"][m] ** 2) * g
    clipped = ((np.abs(d["x"][m]) >= RAIL) | (np.abs(d["y"][m]) >= RAIL) | (np.abs(d["z"][m]) >= RAIL))
    sp = spec_of(t_fc[m], mag, d["rate"], 4096, fmax_hz)
    panels.append((f"campod {key}  ({d['rate']:.0f} Hz fitted, Nyq {d['rate']/2:.0f})",
                   sp, (t_fc[m], clipped), d, d["rate"]))

def draw_lines(ax, alpha=0.85):
    handles = []
    for m in sorted(esc_by_motor):
        e = esc_by_motor[m]
        w = (e.t_s >= fc_t0) & (e.t_s <= fc_t1)
        for mult, ls, lw in ((1, "-", 1.4), (2, "--", 1.1)):
            ax.plot(e.t_s.values[w], e.RPM.values[w] / 60.0 * mult, ls, lw=lw,
                    color=MOTOR_COLORS[m], alpha=alpha,
                    label=f"{m} {'rev' if mult==1 else 'blade-pass'}" if mult <= 2 else None)
    return handles

n = len(panels)
fig, axes = plt.subplots(n + 3, 1, figsize=(15, 3.0 * n + 7.5),
                         gridspec_kw=dict(height_ratios=[3] * n + [1.2, 1.2, 1.2]), sharex=True)
for i, (title, sp, clipinfo, d, rate) in enumerate(panels):
    ax = axes[i]
    if sp is None:
        ax.text(0.5, 0.5, "no usable run", ha="center", transform=ax.transAxes); continue
    f, tt, S, gaps = sp
    dB = 10 * np.log10(np.maximum(S, 1e-12))
    # Scale each panel to its OWN distribution. A fixed dB span saturates any instrument whose
    # noise floor sits close to its peak -- which is the campod sensors, because they are on the
    # structure rather than behind isolators, so their floor is high in absolute terms.
    vmin, vmax = (float(np.percentile(dB, 45)), float(np.percentile(dB, 99.8)))
    ax.pcolormesh(tt, f, dB, cmap=SPEC_CMAP, norm=Normalize(vmin, vmax), shading="nearest", rasterized=True)
    if rate / 2 < fmax_hz:
        ax.axhline(rate / 2, color="#b3211f", lw=1.0, ls=":")
        ax.annotate(f"Nyquist {rate/2:.0f} Hz -- nothing above this line is measured",
                    (tt[0], rate / 2), xytext=(4, 3), textcoords="offset points",
                    fontsize=7, color="#b3211f")
    draw_lines(ax)
    for a, b in gaps:
        ax.axvspan(a, b, color="#ffffff", alpha=0.55, lw=0)
    for tv, lab in ((T_ARM, "ARM"), (LIFT, "liftoff"), (LAND, "touchdown"), (T_DIS, "DISARM")):
        ax.axvline(tv, color="#ffffff", lw=1.0, alpha=0.8)
        ax.annotate(lab, (tv, f[-1]), xytext=(2, -11), textcoords="offset points",
                    fontsize=7, color="#ffffff", rotation=90, va="top")
    ax.set_ylabel("Hz"); ax.set_ylim(0, fmax_hz)
    ax.set_title(title, fontsize=9, loc="left")
    if clipinfo is not None:
        tc, cl = clipinfo
        edges = np.arange(fc_t0, fc_t1, 0.5)
        k = np.clip(np.digitize(tc, edges) - 1, 0, len(edges) - 1)
        frac = np.array([cl[k == j].mean() if (k == j).any() else 0.0 for j in range(len(edges))])
        if frac.max() > 0:
            axc = ax.inset_axes([0, -0.16, 1, 0.13], sharex=ax)
            axc.fill_between(edges, 0, frac * 100, step="mid", color="#b3211f", lw=0)
            axc.set_ylim(0, max(frac.max() * 100 * 1.2, 0.5)); axc.set_yticks([])
            axc.set_ylabel("clip %", fontsize=6, rotation=0, ha="right", va="center")
            axc.tick_params(labelbottom=False, length=0)
            for sp_ in axc.spines.values(): sp_.set_visible(False)
            axc.text(1.002, 0.5, f"max {frac.max()*100:.2f}%", transform=axc.transAxes,
                     fontsize=6, va="center", color="#b3211f")

axT, axR, axC = axes[n], axes[n + 1], axes[n + 2]
axT.plot(ct_w.t_s.values, ct_w.ThO.values * 100, color="#52514e", lw=1.6)
axT.set_ylabel("throttle\n%", fontsize=8)
for m in sorted(esc_by_motor):
    e = esc_by_motor[m]; w = (e.t_s >= fc_t0) & (e.t_s <= fc_t1)
    axR.plot(e.t_s.values[w], e.RPM.values[w], lw=1.6, color=MOTOR_COLORS[m])
    axC.plot(e.t_s.values[w], e.Curr.values[w], lw=1.6, color=MOTOR_COLORS[m])
    # Direct labels: two palette slots are under 3:1 against a light surface, so identity is
    # never carried by colour alone. Anchor them inside the armed window -- at the right edge
    # every motor has spun down to zero and the four labels land on top of each other.
    wl = w & (e.t_s >= T_DIS - 12) & (e.t_s <= T_DIS - 6)
    if wl.any():
        xe = float(e.t_s.values[wl][-1])
        # the front pair (and the rear pair) sit within a few hundred rpm of each other, so a
        # label at the trace's own y superimposes them -- stagger by rank instead
        dy = 9 * (1 if sorted(esc_by_motor).index(m) % 2 == 0 else -1) * (1 + sorted(esc_by_motor).index(m) // 2)
        axR.annotate(f" {m} ({MOTOR_POS[m]})", (xe, e.RPM.values[wl][-1]), fontsize=7,
                     xytext=(3, dy), textcoords="offset points",
                     color=MOTOR_COLORS[m], va="center", ha="left")
        axC.annotate(f" {m}", (xe, e.Curr.values[wl][-1]), fontsize=7,
                     xytext=(3, dy), textcoords="offset points",
                     color=MOTOR_COLORS[m], va="center", ha="left")
axR.set_ylabel("rpm", fontsize=8)
axC.set_title("ESC current is per-ESC uncalibrated (the AM32 generic scale is still in use, "
              "flight-platform.md) -- read shape, not magnitude", fontsize=7, loc="left", color="#52514e")
axC.set_ylabel("ESC\ncurrent A", fontsize=8); axC.set_xlabel("FC time (s)")
for ax in (axT, axR, axC):
    ax.grid(True, **GRID); ax.set_axisbelow(True)
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    for tv in (T_ARM, LIFT, LAND, T_DIS): ax.axvline(tv, color="#a8a7a0", lw=0.8)
axes[0].legend(loc="upper right", fontsize=6, ncol=4, framealpha=0.85)
fig.suptitle(f"{FLIGHT.name} -- vibration spectrum over time, all accelerometers, "
             f"ESC rev (solid) and blade-pass (dashed) overlaid", fontsize=11, y=0.997)
fig.tight_layout(rect=(0, 0, 1, 0.99))
fig.savefig(OUT / "vibration-spectrogram.png", dpi=130, bbox_inches="tight")
print("wrote", OUT / "vibration-spectrogram.png")
plt.show()

## 4. Does each instrument agree, on a window where the source holds still?

The spectrogram above shows lines *tracking*; this quantifies agreement. Agreement is tested
two ways and they are not the same test:

* **Against the RPM prediction** -- involves the ESC telemetry and the pole count, so a
  disagreement could be any of the three.
* **The two pod sensors against each other**, on independently-found peaks, each on its own
  fitted rate, with no prediction and no FC in the loop. This is control #2 from
  `vibration-testing.md` sec. 1, and it is the one an acquisition artifact cannot pass: two
  parts whose clocks differ by ~2.8% can only agree on an absolute frequency if the frequency
  is real.

The window is chosen as the stretch where all motors hold RPM most steadily, because the
residual rev-line sd in that window is the floor on how well anything can agree.

In [ ]:
from scipy.signal import welch

WIN_S = 8.0
cands = []
for a in np.arange(T_ARM + 20, T_DIS - 20 - WIN_S, 1.0):
    sds = [d[(d.t_s >= a) & (d.t_s < a + WIN_S)].RPM.std() for d in esc_by_motor.values()]
    if not any(np.isnan(sds)):
        cands.append((float(np.sum(sds)), float(a)))
cands.sort()
S0, S1 = cands[0][1], cands[0][1] + WIN_S
rev = {m: float(d[(d.t_s >= S0) & (d.t_s <= S1)].RPM.mean() / 60.0) for m, d in esc_by_motor.items()}
sd = {m: float(d[(d.t_s >= S0) & (d.t_s <= S1)].RPM.std() / 60.0) for m, d in esc_by_motor.items()}
print(f"steadiest {WIN_S:.0f} s window: FC t {S0:.1f}-{S1:.1f}")
for m in sorted(rev):
    print(f"  {m} rev {rev[m]:6.2f} Hz  (sd {sd[m]:.2f} Hz -> that is the agreement floor)")
LINE_SEP = min(abs(rev[a] - rev[b]) for a in rev for b in rev if a < b)
HALFBAND = max(0.8, min(2.5, LINE_SEP / 2.5))
print(f"closest pair of rev lines is {LINE_SEP:.2f} Hz apart -> search half-band {HALFBAND:.2f} Hz")
print("  (a half-band wider than half the line spacing lets one motor's line answer for another's)")

spectra = {}
for inst in sorted(F["ACC"].I.unique()):
    a = F["ACC"]; a = a[(a.I == inst) & (a.t_s >= S0) & (a.t_s <= S1)].sort_values("t_s")
    rate = len(a) / (a.t_s.max() - a.t_s.min())
    s = np.sqrt(a.AccX.values ** 2 + a.AccY.values ** 2 + a.AccZ.values ** 2)
    f, P = welch(s - s.mean(), fs=rate, nperseg=min(4096, len(s)), noverlap=min(2048, len(s) // 2))
    spectra[f"FC_ACC{int(inst)}"] = (f, P, rate)
for key, d in accels.items():
    t_fc = align[key]["t_fc"]; m = (t_fc >= S0) & (t_fc <= S1)
    g = d["hdr"]["scale_mg_per_lsb"] / 1000.0 * 9.80665
    s = np.sqrt(d["x"][m] ** 2 + d["y"][m] ** 2 + d["z"][m] ** 2) * g
    f, P = welch(s - s.mean(), fs=d["rate"], nperseg=min(16384, len(s)), noverlap=min(8192, len(s) // 2))
    spectra[key] = (f, P, d["rate"])
    print(f"  {key}: {m.sum():,} samples, bin {f[1]-f[0]:.3f} Hz")

def find_peaks(f, P, fmin, fmax, k=12):
    m = (f >= fmin) & (f <= fmax); fs, Ps = f[m], P[m]
    idx = [i for i in range(2, len(Ps) - 2) if Ps[i] == max(Ps[i - 2:i + 3])]
    return sorted([(float(fs[i]), float(Ps[i])) for i in sorted(idx, key=lambda i: -Ps[i])[:k]])

preds = []
for m in sorted(rev):
    preds.append((f"{m} rev", rev[m]))
for m in sorted(rev):
    preds.append((f"{m} blade-pass", 2 * rev[m]))
names = list(spectra)
print(f"\n{'line':22s} {'pred':>7s} " + " ".join(f"{n:>22s}" for n in names))
dev = {n: [] for n in names}
table = []
for nm, fp in preds:
    cells, row = [], {"line": nm, "pred_hz": round(fp, 2)}
    for n in names:
        f, P, rate = spectra[n]
        if fp > rate / 2:
            cells.append("above Nyquist"); row[n] = None; continue
        band = (f > fp - HALFBAND) & (f < fp + HALFBAND)
        got = float(f[band][np.argmax(P[band])]) if band.any() else float("nan")
        dev[n].append(got - fp); row[n] = round(got, 2)
        cells.append(f"{got:8.2f} ({got-fp:+5.2f})")
    print(f"{nm:22s} {fp:7.2f} " + " ".join(f"{c:>22s}" for c in cells))
    table.append(row)
print("\nrms deviation from the RPM prediction:")
rms = {}
for n in names:
    arr = np.array(dev[n]); rms[n] = float(np.sqrt((arr ** 2).mean()))
    print(f"  {n:24s} {rms[n]:.2f} Hz over {len(arr)} lines   (floor, from RPM wander: "
          f"{np.mean(list(sd.values())):.2f} Hz)")

pod_keys = [k for k in accels]
pair = None
if len(pod_keys) == 2:
    pa = find_peaks(*spectra[pod_keys[0]][:2], 20, fmax_hz)
    pb = find_peaks(*spectra[pod_keys[1]][:2], 20, fmax_hz)
    used, rows = set(), []
    for fa, _ in sorted(pa, key=lambda x: -x[1]):
        c = [(abs(fb - fa), fb) for fb, _ in pb if fb not in used]
        if not c: continue
        dd, fb = min(c)
        if dd < 3.0:
            used.add(fb); rows.append((fa, fb, fb - fa))
    print(f"\ncontrol #2 -- {pod_keys[0]} vs {pod_keys[1]}, independent peaks, own rates, no prediction:")
    print(f"  {'A':>9} {'B':>9} {'delta':>8}")
    for fa, fb, dd in sorted(rows):
        print(f"  {fa:9.2f} {fb:9.2f} {dd:+8.2f} Hz")
    if rows:
        arr = np.array([r[2] for r in rows])
        pair = dict(n=len(rows), mean_hz=float(arr.mean()), rms_hz=float(np.sqrt((arr**2).mean())),
                    max_abs_hz=float(np.abs(arr).max()),
                    peaks=[[round(a,2), round(b,2), round(c,2)] for a,b,c in sorted(rows)])
        print(f"  {len(rows)} matched peaks: mean {arr.mean():+.2f} Hz, rms {np.sqrt((arr**2).mean()):.2f} Hz,"
              f" max |delta| {np.abs(arr).max():.2f} Hz")
        print("  An acquisition artifact cannot pass this: the two clocks differ by "
              f"{(max(accels[k]['rate'] for k in pod_keys)/min(accels[k]['rate'] for k in pod_keys)-1)*100:.2f}%.")

### Amplitude is the measurement; frequency agreement is only the calibration

The sensors *should* agree on where the lines are -- that is what validates the chain. They
should **not** agree on how strong each line is: each sits at a different place on the
structure, so the ratio between them at a given frequency is the transmissibility from one
point to the other. That is the number a mount design actually wants.

Two comparisons, and only one of them is clean:

* **camera-side vs arm-end, same pod** -- same part type, same range, same ODR, same host,
  nothing between them but ~150 mm of carbon. Their ratio *is* structural transmissibility.
  Expect the arm end to be larger: it is nearer the antinode of the arm's first bending mode,
  while the pod body sits near the bolted root (`docs/campod.md`).
* **FC vs pod** -- different part, different range, and the FC sits on isolation bobbins
  *and* behind the IMU's own anti-alias filtering. A ratio here mixes mount isolation with
  sensor filtering and is **not** a transmissibility. Reported, but do not read it as one.

Amplitudes are quoted as the PSD at each line, both converted to m/s^2, so they are
comparable in principle; per-line ratios are given against the camera-side sensor.

In [ ]:
print("accelerometer configuration that bears on amplitude:")
for k in ("INS_ACCEL_FILTER", "INS_GYRO_FILTER", "INS_HNTCH_ENABLE", "INS_HNTCH_MODE",
          "INS_HNTCH_FREQ", "INS_HNTCH_BW", "INS_HNTCH_ATT", "INS_LOG_BAT_MASK"):
    if k in parms:
        print(f"  {k:20s} {parms[k]:g}")
print("  note: INS_ACCEL_FILTER is the EKF-path low-pass; the raw ACC log (LOG_BITMASK bit 19)")
print("  is taken ahead of it, but the sensor's own DLPF still applies. The harmonic notch is a")
print("  CONTROL-path filter -- it does not reduce what the structure does, so it should not")
print("  remove these lines from the raw log.")

def line_amp(name, fp):
    f, P, rate = spectra[name]
    if fp > rate / 2:
        return None
    band = (f > fp - HALFBAND) & (f < fp + HALFBAND)
    return float(P[band].max()) if band.any() else None

REF = [k for k in accels if accels[k]["hdr"]["label"] == "camera"]
REF = REF[0] if REF else list(accels)[0]
print(f"\nPSD at each line (m^2/s^4/Hz), and ratio vs {REF}:")
print(f"{'line':22s} {'Hz':>7s} " + " ".join(f"{n:>24s}" for n in names))
amp_table = []
for nm, fp in preds:
    row = {"line": nm, "hz": round(fp, 2)}
    ref_a = line_amp(REF, fp)
    cells = []
    for n in names:
        a = line_amp(n, fp)
        if a is None:
            cells.append("above Nyquist"); row[n] = None; continue
        r = a / ref_a if ref_a else float("nan")
        row[n] = {"psd": float(f"{a:.4g}"), "ratio_vs_ref": round(r, 3)}
        cells.append(f"{a:10.3g}  x{r:6.2f}")
    print(f"{nm:22s} {fp:7.2f} " + " ".join(f"{c:>24s}" for c in cells))
    amp_table.append(row)

pods = [k for k in accels]
if len(pods) == 2:
    cam = [k for k in pods if accels[k]["hdr"]["label"] == "camera"][0]
    armk = [k for k in pods if accels[k]["hdr"]["label"] == "arm"][0]
    rats = []
    for nm, fp in preds:
        a, c = line_amp(armk, fp), line_amp(cam, fp)
        if a and c: rats.append(a / c)
    rats = np.array(rats)
    print(f"\narm-end / camera-side transmissibility across {len(rats)} lines:")
    print(f"  PSD ratio  median {np.median(rats):.2f}  range {rats.min():.2f}-{rats.max():.2f}")
    print(f"  amplitude ratio (sqrt) median {np.sqrt(np.median(rats)):.2f}")
    print("  Same part type, same ODR, same host -- this one is a structural number.")
    clip_arm = float(((np.abs(accels[armk]["x"]) >= RAIL) | (np.abs(accels[armk]["y"]) >= RAIL) |
                      (np.abs(accels[armk]["z"]) >= RAIL)).mean())
    if clip_arm > 1e-4:
        print(f"  CAVEAT: the arm sensor reaches the +/-{accels[armk]['hdr']['range_g']} g rail on "
              f"{clip_arm*100:.3f}% of samples, so its amplitudes are a LOWER BOUND and its")
        print("  harmonics carry some clipping distortion. +/-16 g is the part's maximum range.")
    transmissibility = dict(pair=[armk, cam], n_lines=int(len(rats)),
                            psd_ratio_median=round(float(np.median(rats)), 3),
                            psd_ratio_min=round(float(rats.min()), 3),
                            psd_ratio_max=round(float(rats.max()), 3),
                            amplitude_ratio_median=round(float(np.sqrt(np.median(rats))), 3),
                            arm_rail_hit_frac=round(clip_arm, 6),
                            arm_amplitudes_are_lower_bound=bool(clip_arm > 1e-4))
else:
    transmissibility = None

## 5. What these frequencies write into a rolling-shutter frame

A line at *f* Hz is written by a rolling shutter as horizontal banding with a row pitch
`rows = (n_rows / readout_s) / f`. The two cameras on this vehicle have **different** readouts,
so the same vibration writes a different pitch in each -- which is why
`still_banding.pitch_to_hz` takes the readout explicitly and defaults to the OAK-D's geometry.
Point it at campod frames without overriding both and the answer is out by ~2.4x.

In [ ]:
CAMERAS = {
    "OAK-D IMX378 12 MP": dict(n_rows=3040, readout_s=0.033),
    "campod IMX708 4608x2592": dict(n_rows=2592, readout_s=2592 * 26.29e-6),
}
for cam, geo in CAMERAS.items():
    print(f"{cam}: {geo['n_rows']} rows / {geo['readout_s']*1000:.1f} ms "
          f"= {geo['n_rows']/geo['readout_s']:.0f} rows/s")
print()
print(f"{'line':22s} {'Hz':>7s} " + " ".join(f"{c:>26s}" for c in CAMERAS))
band_table = []
for nm, fp in preds:
    cells, row = [], {"line": nm, "hz": round(fp, 2)}
    for cam, geo in CAMERAS.items():
        pitch = (geo["n_rows"] / geo["readout_s"]) / fp
        nb = geo["n_rows"] / pitch
        row[cam] = dict(pitch_rows=round(pitch, 1), bands_per_frame=round(nb, 1))
        cells.append(f"{pitch:7.0f} rows, {nb:4.1f} bands")
    print(f"{nm:22s} {fp:7.2f} " + " ".join(f"{c:>26s}" for c in cells))
    band_table.append(row)

## 6. Agent-readable output

In [ ]:
result = {
    "notebook": "vibration-spectrogram.ipynb",
    "input_file": str(LOG),
    "flight": FLIGHT.name,
    "fc": {
        "duration_s": round(fc_duration, 2), "t_arm": round(T_ARM, 2), "t_disarm": round(T_DIS, 2),
        "t_liftoff": round(LIFT, 2), "t_touchdown": round(LAND, 2),
        "arm_to_liftoff_s": round(LIFT - T_ARM, 2),
        "liftoff_arm_gated": True,
        "esc_instance_to_motor": {str(k): v for k, v in sorted(inst2motor.items())},
        "hover_rpm": {m: dict(mean=round(float(d.RPM.mean()), 1), sd=round(float(d.RPM.std()), 1),
                              curr_a=round(float(d.Curr.mean()), 2), position=MOTOR_POS[m])
                      for m, d in sorted(hov.items())},
        "front_rear_split_pct": round(float((front / rear - 1) * 100), 1),
    },
    "instruments": {
        n: dict(rate_hz=round(spectra[n][2], 2), nyquist_hz=round(spectra[n][2] / 2, 1),
                rms_dev_from_rpm_hz=round(rms[n], 2))
        for n in names
    },
    "campod_sensors": {
        k: dict(label=d["hdr"]["label"], device=d["hdr"]["device"],
                odr_nominal_hz=d["hdr"]["odr_hz_nominal"], rate_fitted_hz=round(d["rate"], 2),
                rate_error_pct=round((d["rate"] / d["hdr"]["odr_hz_nominal"] - 1) * 100, 3),
                samples=int(len(d["x"])), truncated_tail_lines=d["bad"],
                self_test_pass=bool(d["hdr"]["self_test"]["all_pass"]),
                rail_hit_frac=round(float(((np.abs(d["x"]) >= RAIL) | (np.abs(d["y"]) >= RAIL) |
                                           (np.abs(d["z"]) >= RAIL)).mean()), 6),
                separation_m=d["hdr"]["separation_m"] or None,
                clock_correction_s=round(align[k]["lag_s"], 3), clock_ncc=round(align[k]["ncc"], 3),
                t_boot_span_s=[round(float(d["t_boot"][0]), 2), round(float(d["t_boot"][-1]), 2)])
        for k, d in accels.items()
    },
    "stationary_window": {"t0": round(S0, 2), "t1": round(S1, 2),
                          "rev_hz": {m: round(v, 2) for m, v in sorted(rev.items())},
                          "rev_sd_hz": {m: round(v, 2) for m, v in sorted(sd.items())},
                          "search_halfband_hz": round(HALFBAND, 2),
                          "line_table": table},
    "two_sensor_control": pair,
    "line_amplitudes": amp_table,
    "transmissibility_arm_over_camera": transmissibility,
    "rolling_shutter_band_pitch": band_table,
    "caveats": [],
}
for k, d in accels.items():
    if not d["hdr"]["separation_m"]:
        result["caveats"].append(f"{k}: separation_m not recorded -- differential-rotation scaling unavailable")
    if d["bad"]:
        result["caveats"].append(f"{k}: {d['bad']} unparseable trailing record(s) -- stream ends unclean")
    rail = float(((np.abs(d["x"]) >= RAIL) | (np.abs(d["y"]) >= RAIL) | (np.abs(d["z"]) >= RAIL)).mean())
    if rail > 1e-4:
        result["caveats"].append(f"{k}: {rail*100:.3f}% of samples at the +/-{d['hdr']['range_g']} g rail "
                                 "-- clipped columns carry clipping harmonics")
    end_fc = float(align[k]["t_fc"][-1])
    if end_fc < T_DIS:
        result["caveats"].append(f"{k}: stream ends at FC t={end_fc:.1f} s, {T_DIS-end_fc:.1f} s before DISARM "
                                 "-- touchdown not covered")
(OUT / "vibration-spectrogram.json").write_text(json.dumps(result, indent=2) + "\n")
print(json.dumps(result, indent=2))